        # 🕷️ L12　網路爬蟲 requests + BeautifulSoup
        **Python 冒險之旅 2026**　｜　Day 6（09/05 六）🏰 資料之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 12 章 12.1–12.5


        ### 🎯 這一關你會學到
        - 用 requests 取得網頁、判讀狀態碼
- 用 BeautifulSoup 的 find / select 解析資料
- 把爬到的資料整理成串列與字典

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L12"
_SALT = "python-quest-2026-datama"
_TASKS = ["12-1", "12-2", "12-3", "12-4", "12-5", "12-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_12_1(run):
    out, ns = run()
    lines = 行列表(out)
    want = ["碁峰暢銷書籍", "優質好書", "Excel VBA基礎必修課", "C語言基礎必修課", "Python基礎必修課", "DTC粉專"]
    missing = [w for w in want if w not in lines]
    return (not missing, f"還缺：{missing}")
任務定義("12-1", _check_12_1, 提示="bs.title.text、bs.find('h3').text、bs.find_all('a')。")

def _check_12_2(run):
    out, ns = run()
    lines = 行列表(out)
    return ("優質好書" in lines and "https://www.facebook.com/dtcbook" in lines, "應該印出 優質好書 與 https://www.facebook.com/dtcbook。")
任務定義("12-2", _check_12_2, 提示="'.blueText' 是 class、'#linkDtc' 是 id；['href'] 取屬性。")

def _check_12_3(run):
    out, ns = run()
    if not 出現(out, "狀態碼：200"): return (False, "狀態碼應該是 200（確認網路連線）。")
    return (出現(out, "勇者書店-本月暢銷榜") and 出現(out, "練習網頁"), "要印出標題與 notice 的文字。")
任務定義("12-3", _check_12_3, 提示="r.status_code、bs.title.text。")

def _check_12_4(run):
    out, ns = run()
    books = ns.get("books", [])
    if len(books) != 8: return (False, f"應該有 8 本書，現在 {len(books)} 本。")
    if not all(isinstance(b.get('價格'), int) for b in books): return (False, "價格要轉成整數。")
    if books[0] != {'書名': '最新 Python 基礎必修課', '價格': 520, '分類': '程式設計'}: return (False, f"第一本應該是 {{'書名': '最新 Python 基礎必修課', '價格': 520, '分類': '程式設計'}}，現在是 {books[0]}。")
    return (出現(out, "最貴的書：深度學習冒險指南890"), "最貴的書是 深度學習冒險指南 890。")
任務定義("12-4", _check_12_4, 提示="price = int(li.select('.price')[0].text[1:])。")

def _check_12_5(run):
    out, ns = run()
    a = ns.get("authors", [])
    if len(a) != 10: return (False, f"第一頁應該有 10 句，現在 {len(a)} 句（確認網路連線）。")
    return (a[0] == "Albert Einstein" and a.count("Albert Einstein") == 3, "第一位作者是 Albert Einstein，而且他出現 3 次。")
任務定義("12-5", _check_12_5, 提示="author = q.select('small.author')[0].text。")

def _check_12_6(run):
    out, ns = run()
    c = ns.get("count", {})
    if c.get("Albert Einstein") != 3: return (False, "Albert Einstein 應該是 3 次。")
    lines = 行列表(out)
    return (lines and lines[0].startswith("Albert Einstein: 3"), "第一行應該是 Albert Einstein: 3。")
任務定義("12-6", _check_12_6, 提示="count[author] = count.get(author, 0) + 1。")


## 🕷️ 12-1　網路爬蟲是什麼？
**爬蟲（web crawler）**＝ 用程式自動「打開網頁 → 讀取內容 → 擷取需要的資料」。兩個主角：

| 套件 | 工作 | 一句話 |
|---|---|---|
| `requests` | 連到網址，把網頁原始碼抓回來 | 「幫我把這一頁拿回來」 |
| `BeautifulSoup`（bs4） | 解析 HTML，用標籤／class／id 找資料 | 「從這一頁裡挑出我要的」 |

> ⚖️ **爬蟲禮儀**：只抓公開資料、看網站的使用條款與 `robots.txt`、不要短時間大量請求、尊重個資。本課練習用自己的練習頁與 [quotes.toscrape.com](https://quotes.toscrape.com/)（專門給爬蟲練習的網站）。

### HTML 長什麼樣？
```html
<h3 class="blueText">優質好書</h3>
<ul>
  <li><a href="http://...">Python 基礎必修課</a></li>
</ul>
<p><a href="..." id="linkDtc">DTC粉專</a></p>
```
`<標籤 屬性="值">內容</標籤>`。**class** 可以重複（像分類標籤），**id** 在一頁中唯一（像身分證）。

In [ ]:
from bs4 import BeautifulSoup
html = """
<html><head><title>碁峰暢銷書籍</title></head>
<body>
<h3 class="blueText">優質好書</h3>
<ul>
  <li><a href="http://books.gotop.com.tw/v_AEL022600">Excel VBA基礎必修課</a></li>
  <li><a href="http://books.gotop.com.tw/v_AEL019900">C語言基礎必修課</a></li>
  <li><a href="http://books.gotop.com.tw/v_AEL025100">Python基礎必修課</a></li>
</ul>
<p><a href="https://www.facebook.com/dtcbook" id="linkDtc" target="_blank">DTC粉專</a></p>
</body></html>
"""
bs = BeautifulSoup(html, 'html.parser')           # 課本 ex12/bs01.py
print(bs.title.text)                              # 標籤的文字
print(bs.find('h3').text)                         # find：第一個符合的標籤
print(bs.find('a', {'target': '_blank'}).text)    # 用屬性篩選
print(bs.select('.blueText')[0].text)             # select：CSS 選擇器（. 是 class）
print(bs.select('#linkDtc')[0]['href'])           # # 是 id；['href'] 取屬性值
for a in bs.find_all('a'):                        # find_all：全部符合的標籤（串列）
    print(a.text, '→', a['href'])

## 12-2　用 `requests` 抓真的網頁
```python
import requests
r = requests.get(網址)
r.status_code     # 200 表示成功；404 找不到；403 被拒絕
r.text            # 網頁原始碼（字串）
r.encoding        # 編碼；中文亂碼時試試 r.encoding = 'utf-8'
```
我們的練習頁：https://johnnychao.github.io/python-quest-2026/practice/books.html

In [ ]:
import requests
from bs4 import BeautifulSoup
url = "https://johnnychao.github.io/python-quest-2026/practice/books.html"
r = requests.get(url)
print('狀態碼：', r.status_code)
r.encoding = 'utf-8'
bs = BeautifulSoup(r.text, 'html.parser')
print('網頁標題：', bs.title.text)
print('第一本書：', bs.select('.book a')[0].text)

### 🎯 任務 12-1　解析本機 HTML

用上面的 `html` 字串（已幫你放在程式格裡）：印出 `<title>` 的文字、`h3` 的文字，以及所有 `<a>` 的**文字**（每行一個）。

In [ ]:
# 🎯 任務 12-1　解析本機 HTML（請保留這一行）
from bs4 import BeautifulSoup
html = """<html><head><title>碁峰暢銷書籍</title></head><body>
<h3 class="blueText">優質好書</h3>
<ul><li><a href="http://books.gotop.com.tw/v_AEL022600">Excel VBA基礎必修課</a></li>
<li><a href="http://books.gotop.com.tw/v_AEL019900">C語言基礎必修課</a></li>
<li><a href="http://books.gotop.com.tw/v_AEL025100">Python基礎必修課</a></li></ul>
<p><a href="https://www.facebook.com/dtcbook" id="linkDtc" target="_blank">DTC粉專</a></p></body></html>"""
bs = BeautifulSoup(html, 'html.parser')
print(???)          # title 文字
print(???)          # h3 文字
for a in ???:
    print(a.text)

In [ ]:
檢查("12-1")   # ◀ 執行這一格，看看任務 12-1 有沒有過關

### 🎯 任務 12-2　select 選擇器

承上 HTML：用 `select()` 印出 class 為 `blueText` 的文字，以及 id 為 `linkDtc` 的 **href 網址**。

In [ ]:
# 🎯 任務 12-2　select 選擇器（請保留這一行）
from bs4 import BeautifulSoup
html = """<html><head><title>碁峰暢銷書籍</title></head><body>
<h3 class="blueText">優質好書</h3>
<p><a href="https://www.facebook.com/dtcbook" id="linkDtc" target="_blank">DTC粉專</a></p></body></html>"""
bs = BeautifulSoup(html, 'html.parser')
print(bs.select(???)[0].text)
print(bs.select(???)[0][???])

In [ ]:
檢查("12-2")   # ◀ 執行這一格，看看任務 12-2 有沒有過關

### 🎯 任務 12-3　抓練習頁：狀態碼與標題

用 `requests.get()` 抓練習頁 `https://johnnychao.github.io/python-quest-2026/practice/books.html`，印出 `狀態碼：200`、`標題：勇者書店 - 本月暢銷榜`，以及 id 為 `notice` 的文字。

In [ ]:
# 🎯 任務 12-3　抓練習頁：狀態碼與標題（請保留這一行）
import requests
from bs4 import BeautifulSoup
url = "https://johnnychao.github.io/python-quest-2026/practice/books.html"
r = requests.get(url)
r.encoding = 'utf-8'
print("狀態碼：", ???)
bs = BeautifulSoup(r.text, 'html.parser')
print("標題：", ???)
print(bs.select('#notice')[0].text)

In [ ]:
檢查("12-3")   # ◀ 執行這一格，看看任務 12-3 有沒有過關

### 🎯 任務 12-4　書單整理成字典串列

從練習頁抓出所有 `.book`：把每本書整理成字典 `{'書名': ..., '價格': 整數, '分類': ...}` 放進串列 `books`，印出每本書，最後印出 `最貴的書：深度學習冒險指南 890`。提示：價格文字是 `$520`，用 `[1:]` 去掉 `$` 再 `int()`。

In [ ]:
# 🎯 任務 12-4　書單整理成字典串列（請保留這一行）
import requests
from bs4 import BeautifulSoup
r = requests.get("https://johnnychao.github.io/python-quest-2026/practice/books.html")
r.encoding = 'utf-8'
bs = BeautifulSoup(r.text, 'html.parser')
books = []
for li in bs.select('.book'):
    title = li.find('a').text
    price = ???
    cat = li.select('.cat')[0].text
    books.append({'書名': title, '價格': price, '分類': cat})
for b in books:
    print(b)
top = max(books, key=lambda b: b['價格'])
print("最貴的書：", top['書名'], top['價格'])

In [ ]:
檢查("12-4")   # ◀ 執行這一格，看看任務 12-4 有沒有過關

### 🎯 任務 12-5　名言網站：抓 10 句名言

抓 https://quotes.toscrape.com/ 第一頁，印出 10 句名言與作者（格式 `作者：名言`），並把作者存進串列 `authors`。網頁結構：每句在 `div.quote` 裡，名言是 `span.text`，作者是 `small.author`。

In [ ]:
# 🎯 任務 12-5　名言網站：抓 10 句名言（請保留這一行）
import requests
from bs4 import BeautifulSoup
r = requests.get("https://quotes.toscrape.com/")
bs = BeautifulSoup(r.text, 'html.parser')
authors = []
for q in bs.select('div.quote'):
    text = q.select('span.text')[0].text
    author = ???
    authors.append(author)
    print(f"{author}：{text}")
print(len(authors), "句")

In [ ]:
檢查("12-5")   # ◀ 執行這一格，看看任務 12-5 有沒有過關

### 🎯 任務 12-6　作者統計（爬蟲 × 字典）

承上，用字典統計每位作者在第一頁出現的次數，依次數由高到低印出（`Albert Einstein: 3` …）。

In [ ]:
# 🎯 任務 12-6　作者統計（爬蟲 × 字典）（請保留這一行）
import requests
from bs4 import BeautifulSoup
r = requests.get("https://quotes.toscrape.com/")
bs = BeautifulSoup(r.text, 'html.parser')
count = {}
for q in bs.select('div.quote'):
    author = q.select('small.author')[0].text
    ???
for name, n in sorted(count.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {n}")

In [ ]:
檢查("12-6")   # ◀ 執行這一格，看看任務 12-6 有沒有過關

## 💡 挑戰題（不計分）
quotes.toscrape.com 有很多頁（`/page/2/`、`/page/3/`…）。用 `for` 迴圈抓前 3 頁，統計所有作者出現次數。記得每次請求之間 `time.sleep(1)` 以示禮貌。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🐼 L13 NumPy／pandas 資料分析初探** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L13_numpy_pandas.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/